# ViT-B/16 — Healthy / Unhealthy Food Classifier (simple)
PyTorch. Frozen ViT backbone + linear head. ~10 min on Colab T4.
Dataset: `maia2000/food-binary-dataset` (35k frames, already binary-labeled).


In [ ]:
# 1. Install + HF login
import os, subprocess, sys
subprocess.run(["pip","-q","install","huggingface_hub","hf_transfer",
                "datasets","scikit-learn"], check=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN").strip()
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

HF_MODEL  = "maia2000/food-classifier-binary-vit"
HF_SOURCE = "ethz/food101"   # public Food-101, 101 classes x 1000 images
print("logged in")


In [ ]:
# 2. Load Food-101 and remap 101 classes -> binary healthy/unhealthy
# No manual upload needed. Food-101 is public on HF.
from datasets import load_dataset

UNHEALTHY = {
    "apple_pie","baby_back_ribs","baklava","beignets","bread_pudding",
    "breakfast_burrito","cannoli","carrot_cake","cheesecake","chocolate_cake",
    "chocolate_mousse","churros","club_sandwich","creme_brulee","croque_madame",
    "cup_cakes","donuts","eggs_benedict","fish_and_chips","french_fries",
    "french_onion_soup","french_toast","fried_calamari","fried_rice","garlic_bread",
    "grilled_cheese_sandwich","hamburger","hot_dog","ice_cream","lasagna",
    "lobster_roll_sandwich","macaroni_and_cheese","macarons","nachos","onion_rings",
    "pancakes","panna_cotta","pizza","poutine","pulled_pork_sandwich",
    "ravioli","red_velvet_cake","spaghetti_bolognese","spaghetti_carbonara",
    "strawberry_shortcake","takoyaki","tiramisu","waffles","foie_gras",
    "pork_chop","cheese_plate","chicken_quesadilla","chicken_wings",
    "clam_chowder","crab_cakes","frozen_yogurt","lobster_bisque","samosa",
}
# Everything else in Food-101 is treated as HEALTHY (salads, grilled fish, soups, etc.)

print("Streaming Food-101 from HF (first load caches ~5GB)...")
ds = load_dataset(HF_SOURCE, split="train")       # 75,750 images
classes_src = ds.features["label"].names
print(f"source classes: {len(classes_src)}")

def to_binary(ex):
    name = classes_src[ex["label"]]
    ex["binary"] = 1 if name in UNHEALTHY else 0   # 0=healthy, 1=unhealthy
    return ex
ds = ds.map(to_binary, num_proc=2)

# Split 70/15/15
ds = ds.shuffle(seed=42)
n = len(ds)
n_val, n_test = int(0.15*n), int(0.15*n)
n_train = n - n_val - n_test
train_hf = ds.select(range(n_train))
val_hf   = ds.select(range(n_train, n_train+n_val))
test_hf  = ds.select(range(n_train+n_val, n))
print(f"train={len(train_hf)}  val={len(val_hf)}  test={len(test_hf)}")

# class balance check
import collections
for name, split in [("train",train_hf),("val",val_hf),("test",test_hf)]:
    c = collections.Counter(split["binary"])
    print(f"  {name}: healthy={c[0]}  unhealthy={c[1]}")


In [ ]:
# 3. Wrap HF datasets as PyTorch DataLoaders
import torch, io
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMG = 224
train_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.1, 0.1, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

class HFImageDs(Dataset):
    def __init__(self, hf_ds, tf):
        self.ds, self.tf = hf_ds, tf
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        ex = self.ds[i]
        img = ex["image"]
        if not isinstance(img, Image.Image):
            img = Image.open(io.BytesIO(img["bytes"]))
        return self.tf(img.convert("RGB")), ex["binary"]

train_ds = HFImageDs(train_hf, train_tf)
val_ds   = HFImageDs(val_hf,   eval_tf)
test_ds  = HFImageDs(test_hf,  eval_tf)

BS = 64 if torch.cuda.is_available() else 8
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)

# For reporting
class FullShim:
    classes = ["healthy", "unhealthy"]
full = FullShim()
print(f"classes: {full.classes}  batch: {BS}")


In [ ]:
# 4. Train — ViT-B/16 frozen, 10 epochs, AMP, early stopping
import torch, torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev)

model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
for p in model.parameters(): p.requires_grad = False   # freeze backbone
model.heads = nn.Linear(model.hidden_dim, 2)           # 2-class head (trainable)
model = model.to(dev)

opt    = torch.optim.AdamW(model.heads.parameters(), lr=1e-3, weight_decay=1e-4)
crit   = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler("cuda") if dev.type == "cuda" else None

def run_epoch(loader, train):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=dev.type=="cuda"):
            logits = model(x); loss = crit(logits, y)
        if train:
            opt.zero_grad()
            if scaler: scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else: loss.backward(); opt.step()
        loss_sum += loss.item() * x.size(0)
        correct  += (logits.argmax(-1) == y).sum().item()
        total    += x.size(0)
    return loss_sum/total, correct/total

best_val, patience, wait = 0.0, 3, 0
history = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}
for ep in range(1, 11):
    tl, ta = run_epoch(train_loader, train=True)
    with torch.no_grad(): vl, va = run_epoch(val_loader, train=False)
    history["train_loss"].append(tl); history["train_acc"].append(ta)
    history["val_loss"].append(vl);   history["val_acc"].append(va)
    print(f"epoch {ep:2d}  train_loss={tl:.4f} acc={ta:.4f}  val_loss={vl:.4f} acc={va:.4f}")
    if va > best_val:
        best_val = va; wait = 0
        torch.save(model.state_dict(), "/content/best.pth")
    else:
        wait += 1
        if wait >= patience: print("early stop"); break

print(f"\nbest val acc: {best_val:.4f}")


In [ ]:
# 5. Evaluate on test set
import json, numpy as np, torch, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score

model.load_state_dict(torch.load("/content/best.pth", map_location=dev))
model.eval()

ys, yhat, probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(dev)
        p = torch.softmax(model(x), dim=-1).cpu().numpy()
        ys.extend(y.tolist())
        yhat.extend(p.argmax(-1).tolist())
        probs.extend(p[:, 1].tolist())

ys, yhat, probs = map(np.array, (ys, yhat, probs))
acc = float((ys == yhat).mean())
f1  = float(f1_score(ys, yhat, average="macro"))
auc = float(roc_auc_score(ys, probs))
print(f"test acc={acc:.4f}  f1={f1:.4f}  auc={auc:.4f}\n")
print(classification_report(ys, yhat, target_names=full.classes, digits=4))

cm = confusion_matrix(ys, yhat)
fig, ax = plt.subplots(figsize=(4,4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(full.classes); ax.set_yticklabels(full.classes)
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha="center", va="center",
                color="white" if cm[i,j] > cm.max()/2 else "black")
plt.title(f"test confusion (acc={acc:.3f})"); plt.tight_layout()
plt.savefig("/content/confusion.png", dpi=100); plt.show()

metrics = {"classes": full.classes, "accuracy": acc, "macro_f1": f1,
           "roc_auc": auc, "confusion_matrix": cm.tolist(), "history": history}
Path("/content/metrics.json").write_text(json.dumps(metrics, indent=2))


In [ ]:
# 6. Export to mobile — TorchScript (iOS/Android) + ONNX (cross-platform)
import torch
from torch.utils.mobile_optimizer import optimize_for_mobile

model.eval().cpu()
example = torch.randn(1, 3, 224, 224)

# --- TorchScript + mobile optimizer (PyTorch Mobile / ExecuTorch / LibTorch) ---
traced = torch.jit.trace(model, example, check_trace=False)
traced_mobile = optimize_for_mobile(traced)
traced_mobile._save_for_lite_interpreter("/content/model_mobile.ptl")  # .ptl = lite interpreter
traced.save("/content/model.torchscript.pt")
print("TorchScript:", Path("/content/model.torchscript.pt").stat().st_size // 1024, "KB")
print("Lite (mobile):", Path("/content/model_mobile.ptl").stat().st_size // 1024, "KB")

# --- ONNX (Core ML, TFLite via onnx2tf, ONNX Runtime Mobile, etc.) ---
torch.onnx.export(
    model, example, "/content/model.onnx",
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)
print("ONNX:", Path("/content/model.onnx").stat().st_size // 1024, "KB")

# Quick verification: outputs match
import numpy as np
with torch.no_grad():
    torch_out = model(example).numpy()
lite_out = torch.jit.load("/content/model_mobile.ptl")(example).numpy()
assert np.allclose(torch_out, lite_out, atol=1e-4), "mobile output diverged!"
print("verified: mobile output matches PyTorch within tolerance")


In [ ]:
# 7. Push everything to HuggingFace (model + mobile + metrics + card)
from huggingface_hub import HfApi, create_repo

Path("/content/README.md").write_text(f"""---
license: apache-2.0
tags: [image-classification, food, binary-classification, vit, mobile]
---
# Binary Healthy/Unhealthy Food Classifier — ViT-B/16

Frozen ViT-B/16 + linear head. Trained on [{HF_DATASET}](https://huggingface.co/datasets/{HF_DATASET}).

## Test metrics
- accuracy: **{acc:.4f}**
- macro F1: **{f1:.4f}**
- ROC-AUC: **{auc:.4f}**

## Files
| File | Format | Use |
|---|---|---|
| `best.pth` | PyTorch state dict | training / fine-tuning |
| `model.torchscript.pt` | TorchScript | server / LibTorch |
| `model_mobile.ptl` | TorchScript Lite | **iOS / Android** (PyTorch Mobile) |
| `model.onnx` | ONNX | Core ML, TFLite (via onnx2tf), ONNX Runtime Mobile |

## Inference (Python)
```python
import torch, torchvision.transforms as T
from PIL import Image
m = torch.jit.load("model.torchscript.pt").eval()
tf = T.Compose([T.Resize((224,224)), T.ToTensor(),
                T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
img = tf(Image.open("food.jpg").convert("RGB")).unsqueeze(0)
probs = torch.softmax(m(img), dim=-1)[0]
print({{"healthy": probs[0].item(), "unhealthy": probs[1].item()}})
```
""")

api = HfApi(token=HF_TOKEN)
create_repo(HF_MODEL, repo_type="model", token=HF_TOKEN, exist_ok=True)
for f in ["/content/best.pth",
          "/content/model.torchscript.pt",
          "/content/model_mobile.ptl",
          "/content/model.onnx",
          "/content/metrics.json",
          "/content/confusion.png",
          "/content/README.md"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=Path(f).name,
                    repo_id=HF_MODEL, token=HF_TOKEN)
print(f"\n-> https://huggingface.co/{HF_MODEL}")
